### LLM Gateways

It is a smart middleware that exist between your app and LLM provider.
Your app is using different LLM providers, so when a request comes, LLM gateway will be redirecting that particular request to the specific LLM provider and getting the response and the response will be given back to the user.   
Core capabilities:
1. Unified API function
2. Automatic Fallbacks 
3. Smart routing
4. Load Balancing
5. Caching
6. Observability 
7. Guardrails

LiteLLM - open source LLM gateways 

### Part 1: What is an LLM Gateway?
Think of an LLM Gateway as a smart middleware layer that sits between your application and multiple LLM providers (OpenAI, Anthropic, Google, Groq, Cohere, local models, etc.).

                    ┌─────────────────────────────┐
                    │       Your Application      │
                    │  (Chatbot, RAG, Agent, etc) │
                    └──────────────┬──────────────┘
                                   │
                                   ▼
                    ┌─────────────────────────────┐
                    │       LLM GATEWAY           │
                    │  • Routing                  │
                    │  • Fallbacks                │
                    │  • Caching                  │
                    │  • Rate Limiting            │
                    │  • Cost Tracking            │
                    │  • Observability            │
                    └──────┬─────┬─────┬─────┬────┘
                           │     │     │     │
                           ▼     ▼     ▼     ▼
                        OpenAI Claude Gemini Groq
### Without a Gateway (The Pain 😩)
1. Different SDKs and APIs for every provider
2. No fallback if one provider goes down
3. No central place to track costs
4. Hard to switch models without rewriting code
5. No caching → paying twice for the same query

### With a Gateway (The Joy 😎)
1. One unified API for 100+ providers
2. Automatic fallbacks if a provider fails
3. Centralized logging, cost tracking, rate limiting
4. Swap models with a config change, no code rewrite
5. Cache repeated queries → save money


In [1]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

## import LiteLLM normally
from litellm import completion

In [2]:
import litellm
litellm.suppress_debug_info = True

In [3]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)


In [35]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

groq_key = os.getenv("GROQ_API_KEY")

print("OPENAI api key loaded", "yes" if os.getenv("OPENAI_API_KEY") else "no")
print("GOOGLE api key loaded", "yes" if os.getenv("GOOGLE_API_KEY") else "no")
print("GROQ api key loaded", "yes" if os.getenv("GROQ_API_KEY") else "no")
if groq_key:
    print(f"Groq API key: {groq_key[:8]}...{groq_key[-4:]}")
else:
    print("❌ GROQ_API_KEY not found")

OPENAI api key loaded yes
GOOGLE api key loaded yes
GROQ api key loaded yes
Groq API key: gsk_wrLm...hsFt


## PART 3: The Simplest LiteLLM example - unified API

the biggest pain point: every provider has a different SDK.
LiteLLM gives you one function - completion() - that works with all of them. Look at how clean this is:

In [ ]:
from litellm import completion

## same code , different providers, - just change the model string

## call openai

response_openai = completion(
    model="gpt-4o-mini",
    messages =[{"role":"user", "content":"Explain RAG in one sentence"}]
)
print("Openai:", response_openai.choices[0].message.content)

response_groq = completion(
    model="groq/llama-3.3-70b-versatile",
    messages =[{"role":"user", "content":"Explain RAG in one sentence"}]
)
print("GROQ:", response_groq.choices[0].message.content)

In [5]:
from litellm import completion

prompt="Explain RAG in one sentence"

providers = [
    ("OpenAI", "gpt-4o-mini"),
    ("Groq", "groq/llama-3.3-70b-versatile"),
    ("Antropic", "claude-3-5-haiku-20241022"),
    ("Gemini", 'gemin/gemini-1.5-flash')
]

#one loop, one function call, multiple provdiers

for label, model in providers:
    try:
        r = completion(model=model, message=[{"role": "user", "contnet": prompt}])
        print(f"{label:<15}: {r.choices[0].message.content[:80]}")
    except Exception as e:
        print(f"{label:<15}: {type(e).__name__}")


OpenAI         : RateLimitError
Groq           : BadRequestError
Antropic       : BadRequestError
Gemini         : BadRequestError


In [ ]:
from litellm import completion

## define a fallback chain: try GPT first, then claude, then Groq
response = completion(
    model="gemini/gemini-3.5-flash",
    messages=[{"role":"user", "content":"What is LLM gateway"}],
    fallback={
        "gpt-4o-mini",
        'groq/llama-3.3-70b-versatile'
    }
)

print("Response", response.choices[0].message.content[:200], '...')
print("\nWhich model actually answered", response.model)

Response An **LLM Gateway** (Large Language Model Gateway) is a specialized management layer (or middleware) that sits between your applications and the various LLM providers (like OpenAI, Anthropic, Cohere, o ...

Which model actually answered gemini-3.5-flash


### Cost Tracking - Know where your money goes
LiteLLM automatically calculates the cost of every call using its built-in pricing database. No more surprise bills. 

In [ ]:
from litellm import completion, completion_cost

response = completion(
    model="gemini-3.5-flash",
    messages=[{
        "role":"user",
        "content":"Write a haiku about AI."
    }]
)

## Get the exact USD cost of this single cell

cost = completion_cost(completion_response = response)
print("Response", response.choices[0].message.content)
print("\nInput tokens", response.usage.prompt_token)
print("\nOutput tokens", response.usage.output_tokens)
print(f"\nCost: ${cost:.8f}")

### Part 6: Caching
If 100 users ask "What is RAG?", you dont need to call LLM 100 times. 
Enable in mmemory caching with one line

In [22]:
import litellm

## Rest any callback/strategies left over from earlier cells
litellm.callbacks = []
litellm.success_callback = []
litellm.failure_callback = []
litellm._async_success_callback = []
litellm._async_failure_callback = []

## also clear any router-strategy state

litellm.cache = None

print("LiteLLM state reset - ready for clean caching")

LiteLLM state reset - ready for clean caching


In [40]:
import os
import time
import litellm

from dotenv import load_dotenv
from litellm import completion
from litellm.caching import Cache

load_dotenv()

# Enable in-memory caching
litellm.cache = Cache(type="local")

prompt = "What does LLM stand for? Answer in one line"

# First call - hits Groq API
start = time.time()

r1 = completion(
    model="groq/openai/gpt-oss-120b",
    messages=[
        {"role": "user", "content": prompt}
    ],
    caching=True
)

t1 = time.time() - start

print(
    f"First call (API): {t1:.2f}s - "
    f"{r1.choices[0].message.content}"
)


# Second call - should come from cache
start = time.time()

r2 = completion(
    model="groq/openai/gpt-oss-120b",
    messages=[
        {"role": "user", "content": prompt}
    ],
    caching=True
)

t2 = time.time() - start

print(
    f"Second call (cache): {t2:.2f}s - "
    f"{r2.choices[0].message.content}"
)

print(
    f"\nSpeedup: {t1/t2:.1f}x faster"
)

First call (API): 0.51s - LLM stands for Large Language Model.
Second call (cache): 0.01s - LLM stands for Large Language Model.

Speedup: 41.5x faster


In [39]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

print("Status:", response.status_code)

if response.ok:
    models = response.json()["data"]

    print("\nAvailable models:\n")

    for model in models:
        print(model["id"])
else:
    print(response.text)
    
available_models = [
    model["id"]
    for model in models
    if model.get("active")
]

print("AVAILABLE MODELS", available_models)

Status: 200

Available models:

allam-2-7b
openai/gpt-oss-120b
meta-llama/llama-prompt-guard-2-86m
whisper-large-v3-turbo
groq/compound
openai/gpt-oss-safeguard-20b
openai/gpt-oss-20b
groq/compound-mini
canopylabs/orpheus-arabic-saudi
whisper-large-v3
qwen/qwen3.6-27b
meta-llama/llama-prompt-guard-2-22m
canopylabs/orpheus-v1-english
AVAILABLE MODELS ['allam-2-7b', 'openai/gpt-oss-120b', 'meta-llama/llama-prompt-guard-2-86m', 'whisper-large-v3-turbo', 'groq/compound', 'openai/gpt-oss-safeguard-20b', 'openai/gpt-oss-20b', 'groq/compound-mini', 'canopylabs/orpheus-arabic-saudi', 'whisper-large-v3', 'qwen/qwen3.6-27b', 'meta-llama/llama-prompt-guard-2-22m', 'canopylabs/orpheus-v1-english']


### Smart Routing - The right model for the right job

**Why use one model for everything?**
- Coding tasks - Claude sonnet
- Cheap Summaries - GPT-4o-mini
- Super fast replies - GROQ Llama
- Complex reasoning - Claude Opus

Use LiteLLM's **Router** to define routing rules:

In [42]:
import os
from litellm import Router

model_list = [
    {
        "model_name": "fast-cheap",
        "litellm_params": {
            "model":"groq/openai/gpt-oss-120b",
            "api_key": os.getenv("GROQ_API_KEY")
        }
    },
    {
        "model_name": "smart-coding",
        "litellm_params": {
            "model":"gpt-4o",
            "api_key": os.getenv("OPENAI_API_KEY")
        }
    },
    {
        "model_name": "balanced",
        "litellm_params": {
            "model":"gpt-4o-mini",
            "api_key": os.getenv("OPENAI_API_KEY")
        }
    },
]

router = Router(model_list = model_list)

fast_response = router.completion(
    model='fast-cheap',
    messages=[{"role":"user", "content":"Summarise AI is changing software"}]
)

# code_response = router.completion(
#     model="smart-coding",
#     messages=[{'role':"user", "content":"Write a Python function to reverse a string"}]
# )

print("Fast-cheap (Groq)", fast_response.choices[0].message.content)
# print("\nSmart coding,", code_response.choices[0].message.content)

Fast-cheap (Groq) **How AI Is Transforming Software Development**

| Area | Traditional Approach | AI‑Enhanced Approach | Impact |
|------|----------------------|----------------------|--------|
| **Code Generation** | Developers write most code manually; libraries and templates are reused. | Large language models (e.g., GPT‑4, Claude, Gemini) generate boilerplate, suggest implementations, and even produce full modules from natural‑language prompts. | Speeds up prototyping, reduces repetitive work, expands accessibility for non‑experts. |
| **Testing & QA** | Test cases are handcrafted; bugs are found through manual debugging or scripted test suites. | AI creates test suites, performs automated fuzzing, predicts flaky tests, and pinpoints root‑cause defects using code‑analysis embeddings. | Higher test coverage, faster regression cycles, earlier bug detection. |
| **Debugging & Refactoring** | Debuggers step through code; refactoring is done manually or with limited IDE assistance. | A

### Load Balancing Across Multiple API Keys

Hit rate limits on one OpenAI key? Add more keys to the same alias - the router load-balance automatically.

In [44]:
import os 
from litellm import Router

## two deployments under the same alias
## A pool of 'smart' models - all equally capable , just different providers

model_list = [
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "gpt-4o",
            "api_key": os.getenv("OPENAI_API_KEY"),
        },
        "model_info": {"id": "openai-gpt4o"}
    },
    
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "groq/openai/gpt-oss-120b",
            "api_key": os.getenv("GROQ_API_KEY"),
        },
        "model_info": {"id": "groq-llama-70b"}
    },
]

router = Router(
    model_list = model_list, 
    routing_strategy="simple-shuffle"
)

print(f"{'Request':<10}{"Deployment Picked":<22}{'Latency':<12}{'Response':<40}")
print("-"*70)


for i in range(6):
    r = router.completion(
        model="gpt-pool",
        messages=[{'role':'user', 'content':f"say Hello, request {i+1}"}]
    )
    ## Pull out which deployment served this request
        
    deplopyment_id = r._hidden_params.get('model_id', 'unknown')
    latency = r._response_ms
    answer = r.choices[0].message.content[:35]
    print(f'{i+1:<9}{deplopyment_id:<22}{latency:>6.0f}ms {answer}')    


Request   Deployment Picked     Latency     Response                                
----------------------------------------------------------------------
1        groq-llama-70b           525ms Hello.
2        groq-llama-70b           721ms Hello! How can I help you with requ
3        groq-llama-70b           734ms Hello, request 3
4        groq-llama-70b           675ms Hello, request 4.
5        groq-llama-70b           662ms Hello, request 5.
6        groq-llama-70b           840ms Hello! Could you let me know what y


### Strategy 1: Least Busy
The "Express Checkout" pattern 
The idea: like picking the shortest line at a supermarket. The router tracks how many requests are currenlty in the flight to each deployment and sends the new request to whichever one is least busy.

In [46]:
import os 
from litellm import Router
from collections import Counter

model_list = [
    {
        "model_name":"chat",
        "litellm_params":{
            "model":"gpt-4o-mini",
            "api_key": os.getenv("OPENAI_API_KEY")
        },
        "model_info": {
            "id":"OpenAI"    
        },
    },
    {
        "model_name":"chat",
        "litellm_params":{
            "model":"groq/openai/gpt-oss-120b",
            "api_key": os.getenv("GROQ_API_KEY")
        },
        "model_info":{
            "id":"GROQ"
        }
    }
]

router = Router(
    model_list = model_list,
    routing_strategy="least-busy"
)

hits = Counter()
for i in range(8):
    r = router.completion(
        model="chat",
        messages=[{"role":"user", 'content':f"Say 'OK' #{i}"}],
        max_tokens=5
    )
    hits[r._hidden_params.get("model_id", "?")] += 1
    print(f"Request {i+1} -> {r._hidden_params.get('model_id', "?")}")

print("\n Distribution: ")
for k,v in hits.most_common():
    print(f"   {k}: {'^'* v}({v})")
    

Request 1 -> GROQ
Request 2 -> GROQ
Request 3 -> GROQ
Request 4 -> GROQ
Request 5 -> GROQ
Request 6 -> GROQ
Request 7 -> GROQ
Request 8 -> GROQ

 Distribution: 
   GROQ: ^^^^^^^^(8)


### Strategy 2: latency-based-routing

The "Always Pick the Fastest" pattern
The idea: the router measures the response time of each deployment over recent calls and sends new requests to whichever has been fastest. Speed wins

In [47]:
import os
from litellm import Router
import time

model_list = [
    {"model_name": "chat",
     "litellm_params": {"model": "gemini-3.5-flash",
                        "api_key": os.getenv("GOOGLE_API_KEY")},
     "model_info": {"id": "🔵 Google Gemini-3.5-flash"}},
    {"model_name": "chat",
     "litellm_params": {"model": "groq/openai/gpt-oss-120b",
                        "api_key": os.getenv("GROQ_API_KEY")},
     "model_info": {"id": "🟢 Groq Llama-3.3"}},
    
]

router = Router(
    model_list=model_list,
    routing_strategy="latency-based-routing"   # 👈 picks the fastest
)

# Send 10 requests and watch which deployments get picked over time
print(f"{'Req':<6}{'Deployment':<32}{'Latency':<10}")
print("-" * 50)

for i in range(10):
    start = time.time()
    r = router.completion(
        model="chat",
        messages=[{"role": "user", "content": "Reply with exactly: OK"}],
        max_tokens=5
    )
    latency_ms = (time.time() - start) * 1000
    deployment = r._hidden_params.get("model_id", "?")
    print(f"#{i+1:<5}{deployment:<32}{latency_ms:>6.0f} ms") 

Req   Deployment                      Latency   
--------------------------------------------------


13:36:10 - LiteLLM:ERROR: vertex_llm_base.py:609 - Failed to load vertex credentials. Check to see if credentials containing partial/invalid information. Error: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.
Traceback (most recent call last):
  File "/Users/muskandawar/codebase/llm_guardrails_evals_gateways/.venv/lib/python3.14/site-packages/litellm/llms/vertex_ai/vertex_llm_base.py", line 605, in get_access_token
    _credentials, credential_project_id = self.load_auth(
                                          ~~~~~~~~~~~~~~^
        credentials=credentials, project_id=project_id
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/Users/muskandawar/codebase/llm_guardrails_evals_gateways/.venv/lib/python3.14/site-packages/litellm/llms/vertex_ai/vertex_llm_base.py", line 163, in load_auth
    creds, creds_project_id = self._credentials_

#1    🟢 Groq Llama-3.3                 12606 ms


13:36:20 - LiteLLM:ERROR: vertex_llm_base.py:609 - Failed to load vertex credentials. Check to see if credentials containing partial/invalid information. Error: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.
Traceback (most recent call last):
  File "/Users/muskandawar/codebase/llm_guardrails_evals_gateways/.venv/lib/python3.14/site-packages/litellm/llms/vertex_ai/vertex_llm_base.py", line 605, in get_access_token
    _credentials, credential_project_id = self.load_auth(
                                          ~~~~~~~~~~~~~~^
        credentials=credentials, project_id=project_id
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/Users/muskandawar/codebase/llm_guardrails_evals_gateways/.venv/lib/python3.14/site-packages/litellm/llms/vertex_ai/vertex_llm_base.py", line 163, in load_auth
    creds, creds_project_id = self._credentials_

APIConnectionError: litellm.APIConnectionError: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.
Traceback (most recent call last):
  File "/Users/muskandawar/codebase/llm_guardrails_evals_gateways/.venv/lib/python3.14/site-packages/litellm/main.py", line 3512, in completion
    model_response = vertex_chat_completion.completion(  # type: ignore
        model=model,
    ...<17 lines>...
        extra_headers=headers,
    )
  File "/Users/muskandawar/codebase/llm_guardrails_evals_gateways/.venv/lib/python3.14/site-packages/litellm/llms/vertex_ai/gemini/vertex_and_google_ai_studio_gemini.py", line 2894, in completion
    _auth_header, vertex_project = self._ensure_access_token(
                                   ~~~~~~~~~~~~~~~~~~~~~~~~~^
        credentials=vertex_credentials,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        project_id=vertex_project,
        ^^^^

### Strategy 4: cost-based-routing — The "Always Cheapest" Pattern
The idea: Pick the deployment that costs the least per token right now. Beautiful for cost-sensitive apps.

In [ ]:
import os
from litellm import Router

# Different providers with very different price points
model_list = [
    {"model_name": "chat",
     "litellm_params": {"model": "gpt-4o",             # ~$2.50/M input tokens
                        "api_key": os.getenv("OPENAI_API_KEY")},
     "model_info": {"id": "🔵 GPT-4o (premium)"}},
    {"model_name": "chat",
     "litellm_params": {"model": "gpt-4o-mini",        # ~$0.15/M input tokens
                        "api_key": os.getenv("OPENAI_API_KEY")},
     "model_info": {"id": "🔵 GPT-4o-mini (cheap)"}},
    {"model_name": "chat",
     "litellm_params": {"model": "groq/llama-3.3-70b-versatile",   # ~$0.05/M
                        "api_key": os.getenv("GROQ_API_KEY")},
     "model_info": {"id": "🟢 Groq Llama (cheapest)"}},
]

router = Router(
    model_list=model_list,
    routing_strategy="simple-shuffle"   # 👈 valid strategy
)

for i in range(5):
    r = router.completion(
        model="chat",
        messages=[{"role": "user", "content": "Hi"}],
        max_tokens=10
    )
    print(f"Request {i+1} → {r._hidden_params.get('model_id', '?')}")

### Part 9: Observability — Log Every Single Call
In production, you must log every LLM call: prompt, response, latency, cost, user_id, etc.

LiteLLM supports custom callbacks — here's a simple logger:

In [48]:
import litellm
from litellm import completion

## A single in-memory log store
call_logs = []

def log_success(kwargs, completion_response, start_time, end_time):
    """ Called automatically after every successful LLM Call """
    call_logs.append({
        "model": kwargs.get("model"),
        "prompt": kwargs["messages"][-1]["content"][:60],
        "input_tokens": completion_response.usage.prompt_tokens,
        "output_tokens": completion_response.usage.completion_tokens,
        "latency_sec": round((end_time - start_time).total_seconds(), 2),
        "cost_usd": kwargs.get("response_cost", 0),
        "user": kwargs.get("user", "anonymous")
    })

def log_failure(kwargs, completion_response, start_time, end_time):
    print("❌ Call failed:", kwargs.get("exception"))
    
# Register the callbacks
litellm.success_callback = [log_success]
litellm.failure_callback = [log_failure]

# Make a few tagged calls
for q, user in [
    ("What is RAG?", "krish"),
    ("Explain transformers.", "student_42"),
    ("What is fine-tuning?", "krish"),
]:
    completion(
        model="groq/openai/gpt-oss-120b",
        messages=[{"role": "user", "content": q}],
        user=user  # tag the call for attribution
    )

# Review the audit log
import json
print(json.dumps(call_logs, indent=2, default=str))

[
  {
    "model": "openai/gpt-oss-120b",
    "prompt": "What is RAG?",
    "input_tokens": 76,
    "output_tokens": 3037,
    "latency_sec": 7.03,
    "cost_usd": 0.0018336,
    "user": "krish"
  },
  {
    "model": "openai/gpt-oss-120b",
    "prompt": "Explain transformers.",
    "input_tokens": 74,
    "output_tokens": 3072,
    "latency_sec": 6.85,
    "cost_usd": 0.0018543,
    "user": "student_42"
  }
]


### Part 10: Integrating the Gateway with LangChain
Here's where it really clicks for production GenAI apps:

**LangChain** for the orchestration (agents, chains, RAG) + **LiteLLM** as the unified LLM backend.

LangChain has a built-in ChatLiteLLM wrapper — drop it in like any other chat model.

In [49]:
from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Build a chat model that talks through LiteLLM
llm = ChatLiteLLM(model="groq/openai/gpt-oss-120b", temperature=0.3)

# A standard LangChain prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI tutor named KrishGPT. Be concise."),
    ("user", "{question}")
])

# Compose with LCEL — same syntax as native LangChain
chain = prompt | llm | StrOutputParser()

answer = chain.invoke({"question": "What is an LLM Gateway in 3 bullets?"})
print(answer)

- **Interface layer** that routes user requests to one or multiple large language models (LLMs) and returns the model’s response.  
- **Abstraction & management** of model selection, versioning, authentication, rate‑limiting, and logging, so developers can switch or combine models without changing their application code.  
- **Enrichment & safety** features such as prompt templating, content filtering, caching, and usage analytics that make LLM integration reliable and secure.


### Part 11: A Real Example — Multi-Provider LangChain Chain with Fallbacks
Let's combine everything: a LangChain chain that uses Claude as primary, with GPT and Groq as fallbacks — and logs every call.

In [ ]:
from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Primary model
primary = ChatLiteLLM(model="gpt-x")

# Fallbacks (any LangChain-compatible model)
fallback_1 = ChatLiteLLM(model="gpt-4o-mini", temperature=0.2)
fallback_2 = ChatLiteLLM(model="groq/llama-3.3-70b-versatile", temperature=0.2)

# LangChain's .with_fallbacks() chains them together
robust_llm = primary.with_fallbacks([fallback_1, fallback_2])

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert AI engineer. Always reply in JSON: {{\"answer\": ...}}"),
    ("user", "{question}")
])

chain = prompt | robust_llm | StrOutputParser()

result = chain.invoke({"question": "What are the top 3 benefits of an LLM Gateway?"})
print(result)

### Part 13: A Mini End-to-End Demo — Smart Router for a Chatbot
Let's build a tiny task-aware chatbot that:

1. Decides what kind of question the user is asking (code, summary, general)
2. Routes to the right model accordingly
3. Falls back if the chosen model fails
4. Logs cost and latency

In [ ]:
import litellm 
from litellm import completion, completion_cost

def classify_task(user_query: str) -> str:
    """ Cheap classifier - uses the fastest model to decide routing """ 
    cls = completion(
        model="groq/openai/gpt-oss-120b",
        messages=[{
            "role":"user",
            "content":(
                f"Classify the following query into EXACTLY one word: "
                f"'code', 'summary', or 'general'. Query: {user_query} \n\nAnswer: "
            )
        }],
        max_tokens = 5
    )
    return cls.choices[0].message.content.strip().lower()

def call_with_fallbacks(model_chain, messages):
    """ Try each model in order; return the first one that succeeds. """
    
    last_error = None
    for model in model_chain:
        try :
            return completion(model=model, messages=messages)
        except Exception as e:
            print(f" {model} failed ({type(e).__name__}), trying next .. ")
            last_error = e
            continue
    raise last_error


def smart_chat(user_query: str): 
    """ Routes to the right model based on the task type, with fallbacks. """
    
    task = classify_task(user_query)
    ## each entry is a full chain: [primary, fallback1, fallback2, ...]
    ## Every model name includes its provider prefix (groq/, anthropic/ etc.)
    
    routing = {
        "code":    ["gpt-4o",                     "gpt-4o-mini",   "groq/llama-3.3-70b-versatile"],
        "summary": ["gpt-4o-mini",                "groq/llama-3.3-70b-versatile"],
        "general": ["groq/openai/gpt-oss-120b", "gpt-4o-mini"],
    }
    
    model_chain = routing.get(task, routing["general"])

    start = time.time()
    response = call_with_fallbacks(
        model_chain=model_chain,
        messages=[{"role": "user", "content": user_query}]
    )
    latency = time.time() - start

    try:
        cost = completion_cost(completion_response=response)
        cost_str = f"${cost:.6f}"
    except Exception:
        cost_str = "n/a"

    return {
        "detected_task": task,
        "model_used":    response.model,
        "answer":        response.choices[0].message.content,
        "latency_sec":   round(latency, 2),
        "cost_usd":      cost_str
    }

     

### The Approach — Pure Python Guardrails Inside LiteLLM Callbacks
LiteLLM gives you two callback hooks that are all you need:

- litellm.input_callback — runs before the LLM call (inspect/modify the prompt)
- litellm.success_callback — runs after a successful LLM call (inspect/modify the response)

Inside these hooks, you can do any Python you want — regex, keyword matching, or even another LLM call for classification. No external libraries needed.Let me show you the full guardrail stack with just LiteLLM.

In [ ]:
import re
import litellm
from litellm import completion

# 🎯 PII patterns — simple, fast, no external dependencies
PII_PATTERNS = {
    "EMAIL":       r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",
    "PHONE_IN":    r"(\+91[\-\s]?)?[6-9]\d{9}",                  # Indian mobile
    "PHONE_US":    r"(\+1[\-\s]?)?\(?\d{3}\)?[\-\s]?\d{3}[\-\s]?\d{4}",
    "SSN":         r"\b\d{3}-\d{2}-\d{4}\b",
    "AADHAAR":     r"\b\d{4}\s?\d{4}\s?\d{4}\b",                 # Indian Aadhaar
    "PAN":         r"\b[A-Z]{5}\d{4}[A-Z]\b",                    # Indian PAN
    "CREDIT_CARD": r"\b\d{4}[\s\-]?\d{4}[\s\-]?\d{4}[\s\-]?\d{4}\b",
    "IP_ADDRESS":  r"\b(?:\d{1,3}\.){3}\d{1,3}\b",
}


def redact_pii(text: str):
    """Replace PII in text with placeholders. Returns (clean_text, detected_list)."""
    detected = []
    clean = text
    for label, pattern in PII_PATTERNS.items():
        matches = re.findall(pattern, clean)
        if matches:
            detected.append({"type": label, "count": len(matches)})
            clean = re.sub(pattern, f"<{label}_REDACTED>", clean)
    return clean, detected


def pii_input_guardrail(kwargs):
    """LiteLLM pre-call hook: scrub PII from user messages."""
    messages = kwargs.get("messages", [])
    for msg in messages:
        if msg.get("role") == "user":
            clean, detected = redact_pii(msg["content"])
            if detected:
                print(f"🚨 PII REDACTED: {detected}")
                msg["content"] = clean


# Register the guardrail
litellm.input_callback = [pii_input_guardrail]


# 🧪 Test
user_msg = (
    "Hi, I'm Krish. My email is krish@krishnaik.in, "
    "my Indian mobile is +91-9876543210, my PAN is ABCDE1234F, "
    "and my Aadhaar is 1234 5678 9012. Help me write Python code."
)

response = completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": user_msg}],
    max_tokens=80
)

print("\n💬 LLM Response:")
print(response.choices[0].message.content)

### Guardrail 2 : Prompt Injection Blocking

In [ ]:
import re
import litellm

